# <center> Homework 135

In [12]:
import tensorflow as tf
from importlib import reload
import tf_model
import tf_data
import pandas as pd
import numpy as np
from pathlib import Path
import tensorflow_datasets as tfds

### При Stateful RNN

#### State-ът НЕ се нулира автоматично.

Тоест:

model.predict(seq1)

model.predict(seq2)

seq2 ще започне със state-а, останал от seq1.



#### Златното правило при stateful inference:

Винаги прави:

model.reset_states()

преди нов независим sequence.

### Unrecognized keyword arguments passed to Embedding: {'batch_input_shape': [1, None]}

В по-старите версии можеше да подадеш batch_input_shape директно в слоя.

В Keras 3 това вече не се поддържа по този начин.

При stateful=True ти трябва фиксиран batch size.
Но вече го задаваш през Input слоя.

Правилният начин е:
#### tf.keras.Input(batch_shape=(1, None))

#### AttributeError: 'Sequential' object has no attribute 'reset_states'

In [ ]:
class ResetStatesCallback(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs):
        self.model.reset_states()

Трябва да е така:

In [ ]:
class ResetStatesCallback(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs):
        for layer in self.model.layers:
            if hasattr(layer, 'reset_states'):
                layer.reset_states()

#### При Stateful RNN и трениране с tf.data.Dataset когато се прави batch трябва да има batch(1, drop_remainder=True)

## Task 0
да се тестват примерите в книгата

    да се проследи и тества подоготовката на данните за трениранте на stateful RNN


In [13]:
shakespeare_url = "https://homl.info/shakespeare" # shortcut URL
filepath = tf.keras.utils.get_file("shakespeare.txt", shakespeare_url)

with open(filepath) as f:
    shakespeare_text = f.read()

In [14]:
text_vec_layer = tf.keras.layers.TextVectorization(split="character", standardize="lower")
text_vec_layer.adapt([shakespeare_text])
encoded = text_vec_layer([shakespeare_text])[0]

encoded -= 2
n_tokens = text_vec_layer.vocabulary_size() - 2
dataset_size = len(encoded)

In [15]:
length = 100

In [16]:
def to_dataset_for_stateful_rnn(sequence, length):
    ds = tf.data.Dataset.from_tensor_slices(sequence)
    ds = ds.window(length + 1, shift=length, drop_remainder=True)
    ds = ds.flat_map(lambda window: window.batch(length + 1)).batch(1, drop_remainder=True)
    return ds.map(lambda window: (window[:, :-1], window[:, 1:])).prefetch(1)

def to_dataset_for_stateful_rnn_explained(sequence, length):
    def print_one_item(ds):
        print(next(iter(ds.take(1))))
        print('\n\n')

    print('==== Sequence ====')
    print(sequence)
    print('\n\n')

    ds = tf.data.Dataset.from_tensor_slices(sequence)
    print('==== Step 1: ds = tf.data.Dataset.from_tensor_slices(sequence) ====')
    print_one_item(ds)

    ds = ds.window(length + 1, shift=length, drop_remainder=True)
    print('==== Step 2: ds = ds.window(length + 1, shift=length, drop_remainder=True) ====')
    print_one_item(ds)

    ds = ds.flat_map(lambda window: window.batch(length + 1))
    print('==== Step 3: ds = ds.flat_map(lambda window: window.batch(length + 1)) ====')
    print_one_item(ds)

    ds = ds.batch(1)
    print('==== Step 4: ds = ds.batch(1) ====')
    print_one_item(ds)

    ds = ds.map(lambda window: (window[:, :-1], window[:, 1:])).prefetch(1)
    print('==== Step 5: ds.map(lambda window: (window[:, :-1], window[:, 1:])).prefetch(1) ====')
    print_one_item(ds)

In [17]:
to_dataset_for_stateful_rnn_explained(encoded[:1_000_000], length)

==== Sequence ====
tf.Tensor([19  5  8 ...  0  2  6], shape=(1000000,), dtype=int64)



==== Step 1: ds = tf.data.Dataset.from_tensor_slices(sequence) ====
tf.Tensor(19, shape=(), dtype=int64)



==== Step 2: ds = ds.window(length + 1, shift=length, drop_remainder=True) ====
<_VariantDataset element_spec=TensorSpec(shape=(), dtype=tf.int64, name=None)>



==== Step 3: ds = ds.flat_map(lambda window: window.batch(length + 1)) ====
tf.Tensor(
[19  5  8  7  2  0 18  5  2  5 35  1  9 23 10 21  1 19  3  8  1  0 16  1
  0 22  8  3 18  1  1 12  0  4  9 15  0 19 13  8  2  6  1  8 17  0  6  1
  4  8  0 14  1  0  7 22  1  4 24 26 10 10  4 11 11 23 10  7 22  1  4 24
 17  0  7 22  1  4 24 26 10 10 19  5  8  7  2  0 18  5  2  5 35  1  9 23
 10 15  3 13  0], shape=(101,), dtype=int64)



==== Step 4: ds = ds.batch(1) ====
tf.Tensor(
[[19  5  8  7  2  0 18  5  2  5 35  1  9 23 10 21  1 19  3  8  1  0 16  1
   0 22  8  3 18  1  1 12  0  4  9 15  0 19 13  8  2  6  1  8 17  0  6  1
   4  8  0 14  1  0  

In [18]:
stateful_train_set = to_dataset_for_stateful_rnn(encoded[:1_000_000], length)
stateful_valid_set = to_dataset_for_stateful_rnn(encoded[1_000_000:1_060_000], length)
stateful_test_set = to_dataset_for_stateful_rnn(encoded[1_060_000:], length)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(batch_shape=(1, None)),
    tf.keras.layers.Embedding(input_dim=n_tokens, output_dim=16),
    tf.keras.layers.GRU(128, return_sequences=True, stateful=True),
    tf.keras.layers.Dense(n_tokens, activation="softmax")
])

class ResetStatesCallback(tf.keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs):
        for layer in self.model.layers:
            if hasattr(layer, 'reset_states'):
                layer.reset_states()

model_ckpt = tf.keras.callbacks.ModelCheckpoint(
    "language_models/shakespeare_model_hw135.keras",
    monitor="val_accuracy",
    save_best_only=True)

model.compile(loss="sparse_categorical_crossentropy", optimizer="nadam", metrics=["accuracy"])
history = model.fit(stateful_train_set, validation_data=stateful_valid_set,
epochs=10, callbacks=[ResetStatesCallback(), model_ckpt])

In [19]:
model = tf.keras.models.load_model('language_models/shakespeare_model_hw135.keras')

In [20]:
model.evaluate(stateful_test_set)

553/553 ━━━━━━━━━━━━━━━━━━━━ 18s 29ms/step - accuracy: 0.4994 - loss: 1.6607


2026-02-25 09:55:19.538672: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
/home/zdravko/Machine_Learning_Intern/venv/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


[1.660714030265808, 0.4994405508041382]

In [ ]:
shakespeare_model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Lambda(lambda X: X - 2),
    model
])

In [ ]:
def extend_text(text, model, gen_func, n_chars=50, temperature=1, threshold=0.7, k=3):
  for _ in range(n_chars):
    text += gen_func(text, model, temperature=temperature, threshold=threshold, k=k)

  return text

def print_tensor(text):
  text = text.numpy()[0].decode("utf-8")
  text = text.replace("\\n", "\n")
  print(text)

def next_char_temp(text, model, temperature=1, threshold=None, k=None):
  y_proba = model.predict([text])[0, -1:]
  rescaled_logits = tf.math.log(y_proba) / temperature
  char_id = tf.random.categorical(rescaled_logits, num_samples=1)[0, 0]
  return text_vec_layer.get_vocabulary()[char_id + 2]

In [ ]:
text = tf.constant(["To be or not to b"])
generated = extend_text(text, shakespeare_model, next_char_temp, temperature=0.5)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 187ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━

In [ ]:
print_tensor(generated)

To be or not to be
me a power and love a matter to her hand.

lady 


In [3]:
def to_non_overlapping_windows(sequence, length):
    ds = tf.data.Dataset.from_tensor_slices(sequence)
    ds = ds.window(length + 1, shift=length, drop_remainder=True)
    return ds.flat_map(lambda window: window.batch(length + 1))

def to_batched_dataset_for_stateful_rnn(sequence, length, batch_size=32):
    parts = np.array_split(sequence, batch_size)
    datasets = tuple(to_non_overlapping_windows(part, length) for part in parts)
    ds = tf.data.Dataset.zip(datasets).map(lambda *windows: tf.stack(windows))
    return ds.map(lambda window: (window[:, :-1], window[:, 1:])).prefetch(1)

list(to_batched_dataset_for_stateful_rnn(tf.range(20), length=3, batch_size=2))

2026-02-25 09:07:04.312504: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2026-02-25 09:07:04.633681: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


[(<tf.Tensor: shape=(2, 3), dtype=int32, numpy=
  array([[ 0,  1,  2],
         [10, 11, 12]], dtype=int32)>,
  <tf.Tensor: shape=(2, 3), dtype=int32, numpy=
  array([[ 1,  2,  3],
         [11, 12, 13]], dtype=int32)>),
 (<tf.Tensor: shape=(2, 3), dtype=int32, numpy=
  array([[ 3,  4,  5],
         [13, 14, 15]], dtype=int32)>,
  <tf.Tensor: shape=(2, 3), dtype=int32, numpy=
  array([[ 4,  5,  6],
         [14, 15, 16]], dtype=int32)>),
 (<tf.Tensor: shape=(2, 3), dtype=int32, numpy=
  array([[ 6,  7,  8],
         [16, 17, 18]], dtype=int32)>,
  <tf.Tensor: shape=(2, 3), dtype=int32, numpy=
  array([[ 7,  8,  9],
         [17, 18, 19]], dtype=int32)>)]

## Task 1
имплементацията на класа GRU

    да се поддържа параметър stateful
    model.reset_states() да занулява state-a с вътрешните състояния

## Task 4
класа Embedding да може да се обучава

In [ ]:
reload(tf_data)
from tf_data import Dataset

def to_dataset_for_stateful_rnn_cus(sequence, length):
    ds = Dataset.from_tensor_slices(sequence)
    ds = ds.window(length + 1, shift=length, drop_remainder=True)
    ds = ds.flat_map(lambda window: window.batch(length + 1)).batch(1, drop_remainder=True)
    return ds.map(lambda window: (window[:, :-1], window[:, 1:])).prefetch(1)

In [ ]:
stateful_train_set = to_dataset_for_stateful_rnn_cus(tf.cast(encoded, tf.float32)[:10_000], length)
stateful_valid_set = to_dataset_for_stateful_rnn_cus(tf.cast(encoded, tf.float32)[10_000:13_000], length)

In [ ]:
next(iter(stateful_train_set.take(1)))

(<tf.Tensor: shape=(1, 100), dtype=float32, numpy=
 array([[19.,  5.,  8.,  7.,  2.,  0., 18.,  5.,  2.,  5., 35.,  1.,  9.,
         23., 10., 21.,  1., 19.,  3.,  8.,  1.,  0., 16.,  1.,  0., 22.,
          8.,  3., 18.,  1.,  1., 12.,  0.,  4.,  9., 15.,  0., 19., 13.,
          8.,  2.,  6.,  1.,  8., 17.,  0.,  6.,  1.,  4.,  8.,  0., 14.,
          1.,  0.,  7., 22.,  1.,  4., 24., 26., 10., 10.,  4., 11., 11.,
         23., 10.,  7., 22.,  1.,  4., 24., 17.,  0.,  7., 22.,  1.,  4.,
         24., 26., 10., 10., 19.,  5.,  8.,  7.,  2.,  0., 18.,  5.,  2.,
          5., 35.,  1.,  9., 23., 10., 15.,  3., 13.]], dtype=float32)>,
 <tf.Tensor: shape=(1, 100), dtype=float32, numpy=
 array([[ 5.,  8.,  7.,  2.,  0., 18.,  5.,  2.,  5., 35.,  1.,  9., 23.,
         10., 21.,  1., 19.,  3.,  8.,  1.,  0., 16.,  1.,  0., 22.,  8.,
          3., 18.,  1.,  1., 12.,  0.,  4.,  9., 15.,  0., 19., 13.,  8.,
          2.,  6.,  1.,  8., 17.,  0.,  6.,  1.,  4.,  8.,  0., 14.,  1.,
          0

In [ ]:
reload(tf_model)
from tf_model import *

model = Sequential([
    Input([None], batch_size=1),
    Embedding(input_dim=n_tokens, output_dim=16),
    GRU(128, return_sequences=True, stateful=True),
    Dense(n_tokens, activation="softmax")
])

class ResetStatesCallback(Callback):
    def on_epoch_begin(self, epoch, logs):
        self.model.reset_states()

model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
init_emb_mat = model.layers[1].embedding_mat.numpy().copy()

history = model.fit(stateful_train_set, validation_data=(stateful_valid_set),
epochs=10, callbacks=[ResetStatesCallback()])

print(np.abs(init_emb_mat - model.layers[1].embedding_mat.numpy()))

Epoch 1/10


0it [00:00, ?it/s]

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, activation, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, activation, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, self, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verb

99it [00:17,  5.74it/s]


    - loss: 8.134 - accuracy: 0.148
    - val_loss: 7.844 - val_accuracy: 0.139
    - Learning Rate: 0.001
Epoch 2/10


...

    - loss: 5.378 - accuracy: 0.394
    - val_loss: 5.905 - val_accuracy: 0.331
    - Learning Rate: 0.001
Epoch 9/10


99it [00:11,  8.44it/s]


    - loss: 5.246 - accuracy: 0.405
    - val_loss: 5.865 - val_accuracy: 0.343
    - Learning Rate: 0.001
Epoch 10/10


99it [00:11,  8.56it/s]


    - loss: 5.120 - accuracy: 0.416
    - val_loss: 5.842 - val_accuracy: 0.354
    - Learning Rate: 0.001
[[2.34286144e-01 3.25147778e-01 2.19755143e-01 1.34366900e-01
  2.64157265e-01 2.64863595e-02 3.58958364e-01 1.31466761e-01
  2.21316338e-01 2.91658312e-01 2.47663468e-01 2.14653373e-01
  1.18665427e-01 7.84686655e-02 6.97451551e-03 1.98644232e-02]
 [5.13545536e-02 5.92498481e-03 1.04841188e-01 5.89067303e-02
  9.92498696e-02 8.51801634e-02 4.49585840e-02 2.51216948e-01
  1.13583595e-01 1.76537201e-01 1.24035403e-01 1.88225657e-02
  5.45531511e-03 1.18668482e-01 6.99743479e-02 2.59563357e-01]
 [9.74976495e-02 9.17321742e-02 2.35779732e-01 6.44640028e-02
  1.24989655e-02 1.26362415e-02 3.22550833e-01 9.02254805e-02
  1.08237177e-01 1.04592837e-01 5.83215766e-02 2.17936009e-01
  1.11091398e-01 2.36766577e-01 1.68334603e-01 2.57354856e-01]
 [1.27122343e-01 9.59912837e-02 2.45699078e-01 8.61298889e-02
  3.26667368e-01 3.47057670e-01 3.02482367e-01 8.31613690e-02
  2.15703875e-01 9.337

## Task 2
класа TextVectorization да може да encode-ва думи

In [ ]:
reload(tf_model)
from tf_model import TextVectorization

text = 'Before, hear me speak. hear'

text_vec = TextVectorization()
text_vec.adapt(text)
text_vec.call(text)

<tf.Tensor: shape=(5,), dtype=int32, numpy=array([3, 2, 4, 5, 2], dtype=int32)>

In [ ]:
text_vec = TextVectorization(split='character')
text_vec.adapt(text)
text_vec.call(text)

<tf.Tensor: shape=(25,), dtype=int32, numpy=
array([ 7,  2,  8,  9,  4,  2,  3,  6,  2,  5,  4,  3, 10,  2,  3, 11, 12,
        2,  5, 13,  3,  6,  2,  5,  4], dtype=int32)>

## Task 3
да се тества примера за sentiment analysis с padding отляво.

In [3]:
raw_train_set, raw_valid_set, raw_test_set = tfds.load(
    name="imdb_reviews",
    split=["train[:90%]", "train[90%:]", "test"],
    as_supervised=True
)

tf.random.set_seed(42)
train_set = raw_train_set.shuffle(5000, seed=42).batch(32).prefetch(1)
valid_set = raw_valid_set.batch(32).prefetch(1)
test_set = raw_test_set.batch(32).prefetch(1)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.C7THZ8_1.0.0/imdb_reviews-train.tfrecor…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.C7THZ8_1.0.0/imdb_reviews-test.tfrecord…

Generating unsupervised examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.C7THZ8_1.0.0/imdb_reviews-unsupervised.…

Dataset imdb_reviews downloaded and prepared to /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.


In [4]:
for review, label in raw_train_set.take(4):
    print()
    rev = review.numpy().decode("utf-8")
    print(rev)
    print("Label:", label.numpy(), 'Length:', len(rev))


This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline. This movie is an early nineties US propaganda piece. The most pathetic scenes were those when the Columbian rebels were making their cases for revolutions. Maria Conchita Alonso appeared phony, and her pseudo-love affair with Walken was nothing but a pathetic emotional plug in a movie that was devoid of any real meaning. I am disappointed that there are movies like this, ruining actor's like Christopher Walken's good name. I could barely sit through it.
Label: 0 Length: 709

I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on the sette and having just eaten a lot. However on this occasion I fell asleep because the film was rubbish. The pl

In [5]:
vocab_size = 1000
text_vec_layer = tf.keras.layers.TextVectorization(max_tokens=vocab_size)
text_vec_layer.adapt(train_set.map(lambda reviews, labels: reviews))

In [6]:
text = next(iter(train_set))[0]
text

<tf.Tensor: shape=(32,), dtype=string, numpy=
array([b'The recent boom of dating show on U. S. television screens has reached a fevered pitch since the first episode of "The Bachelor." Unsuspecting audiences have since been subjected to countless clones and variations, including "The Bachelorette", "Joe Millionaire", "For Love Or Money", and the execrable "Married By America." Hoping to cash in on this trend, and simultaneously tap and exploit a new demographic, Bravo has unleashed the disastrous "Boy Meets Boy" upon the world. And may they have mercy on us all.<br /><br />The premise is simple and is designed to be light-hearted: an eligible gay man is courted by a number of suitors, eliminated show by show until one is left, but there\'s a twist. Half of the men are actually straight. This is not much of a big deal, but the inherent viciousness of the scenario kicks in after hearing the pay-off: if, at the end of the show, the gay man picks a straight man in disguise, the straight ma

In [7]:
text_vec_layer(text)

<tf.Tensor: shape=(32, 433), dtype=int64, numpy=
array([[  2,   1,   1, ...,  46,   5, 300],
       [  1,   3,   2, ...,   0,   0,   0],
       [ 47,  82, 227, ...,   0,   0,   0],
       ...,
       [199,   1,   1, ...,   0,   0,   0],
       [168, 274,   8, ...,   0,   0,   0],
       [  1,   1,   3, ...,   0,   0,   0]])>

In [ ]:
x = text_vec_layer(text)

x = tf.reverse_sequence(
    x,
    seq_lengths=tf.reduce_sum(tf.cast(x != 0, tf.int32), axis=1),
    seq_axis=1
)
x = tf.reverse(x, axis=[-1])
# duma00 -> amud00 -> 00duma
x

<tf.Tensor: shape=(32, 433), dtype=int64, numpy=
array([[  2,   1,   1, ...,  46,   5, 300],
       [  0,   0,   0, ...,  13,   1, 387],
       [  0,   0,   0, ...,   1,   3,   1],
       ...,
       [  0,   0,   0, ...,   1,   1, 231],
       [  0,   0,   0, ..., 352,   1,  23],
       [  0,   0,   0, ..., 116,   1, 122]])>

In [9]:
embed_size = 128
tf.random.set_seed(42)

def left_pad(inp):
    x = tf.reverse_sequence(
        inp,
        seq_lengths=tf.reduce_sum(tf.cast(inp != 0, tf.int32), axis=1),
        seq_axis=1
    )

    return tf.reverse(x, axis=[-1])

model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Lambda(left_pad),
    tf.keras.layers.Embedding(vocab_size, embed_size),
    tf.keras.layers.GRU(128),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(loss="binary_crossentropy", optimizer="nadam", metrics=["accuracy"])
history = model.fit(train_set, validation_data=valid_set, epochs=2)

Epoch 1/2
704/704 ━━━━━━━━━━━━━━━━━━━━ 29s 36ms/step - accuracy: 0.6346 - loss: 0.6297 - val_accuracy: 0.8200 - val_loss: 0.4133
Epoch 2/2
704/704 ━━━━━━━━━━━━━━━━━━━━ 24s 34ms/step - accuracy: 0.8370 - loss: 0.3744 - val_accuracy: 0.8624 - val_loss: 0.3107


In [10]:
model.evaluate(test_set)

782/782 ━━━━━━━━━━━━━━━━━━━━ 14s 18ms/step - accuracy: 0.8670 - loss: 0.3129


[0.3101116120815277, 0.8681600093841553]

In [11]:
embed_size = 128
tf.random.set_seed(42)

model = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Embedding(vocab_size, embed_size),
    tf.keras.layers.GRU(128),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(loss="binary_crossentropy", optimizer="nadam", metrics=["accuracy"])
history = model.fit(train_set, validation_data=valid_set, epochs=2)

Epoch 1/2
704/704 ━━━━━━━━━━━━━━━━━━━━ 26s 34ms/step - accuracy: 0.5009 - loss: 0.6938 - val_accuracy: 0.5016 - val_loss: 0.6934
Epoch 2/2
704/704 ━━━━━━━━━━━━━━━━━━━━ 24s 33ms/step - accuracy: 0.4952 - loss: 0.6951 - val_accuracy: 0.6788 - val_loss: 0.6152


## Task 5
всички RNN класове които сме разработвали да поддържат маскиране, masking proopagation и compute_mask()

In [21]:
reload(tf_data)
from tf_data import Dataset


raw_train_set, raw_valid_set, raw_test_set = tfds.load(
    name="imdb_reviews",
    split=["train[:90%]", "train[90%:]", "test"],
    as_supervised=True
)

tf.random.set_seed(42)
train_set = Dataset(raw_train_set).shuffle(5000, seed=42).batch(32).prefetch(1)
valid_set = Dataset(raw_valid_set).batch(32).prefetch(1)
test_set  = Dataset(raw_test_set).batch(32).prefetch(1)

In [ ]:
reload(tf_model)
from tf_model import *

vocab_size = 1000
text_vec_layer = TextVectorization(max_tokens=vocab_size)
text_vec_layer.adapt(train_set.take(5).map(lambda reviews: reviews[0]))


embed_size = 128
tf.random.set_seed(42)

model = Sequential([
    Input([]),
    text_vec_layer,
    Embedding(vocab_size, embed_size, mask_zero=True),
    GRU(128),
    Dense(1, activation="sigmoid")
])

model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
history = model.fit(train_set.take(30), validation_data=valid_set.take(10), epochs=2)

Epoch 1/2


0it [00:00, ?it/s]

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, self, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, self, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, self, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, activation, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


...

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, metric, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


30it [03:25,  6.85s/it]


    - loss: 0.698 - accuracy: 0.500
    - val_loss: 0.693 - val_accuracy: 0.497
    - Learning Rate: 0.001
Epoch 2/2


30it [02:35,  5.19s/it]


    - loss: 0.694 - accuracy: 0.480
    - val_loss: 0.696 - val_accuracy: 0.456
    - Learning Rate: 0.001


In [22]:
reload(tf_model)
from tf_model import *

vocab_size = 1000
text_vec_layer = TextVectorization(max_tokens=vocab_size)
text_vec_layer.adapt(train_set.take(5).map(lambda reviews: reviews[0]))


embed_size = 128
tf.random.set_seed(42)

model = Sequential([
    Input([]),
    text_vec_layer,
    Embedding(vocab_size, embed_size, mask_zero=True),
    LSTM(128),
    Dense(1, activation="sigmoid")
])

model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
history = model.fit(train_set.take(30), validation_data=valid_set.take(10), epochs=2)

Epoch 1/2


0it [00:00, ?it/s]

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, self, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, self, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, self, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, activation, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


...

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, metric, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


30it [02:54,  5.82s/it]


    - loss: 0.694 - accuracy: 0.509
    - val_loss: 0.699 - val_accuracy: 0.453
    - Learning Rate: 0.001
Epoch 2/2


30it [03:33,  7.12s/it]


    - loss: 0.695 - accuracy: 0.493
    - val_loss: 0.692 - val_accuracy: 0.553
    - Learning Rate: 0.001


## Task 6
класа GRU да може да ражботи с ragged tensors

In [ ]:
reload(tf_model)
from tf_model import *

text_vec_layer_ragged = TextVectorization(max_tokens=vocab_size, ragged=True)
text_vec_layer_ragged.adapt(train_set.take(5).map(lambda reviews: reviews[0]))
text_vec_layer_ragged.call(tf.constant(["Great movie!", "This is DiCaprio's best role."]))

<tf.RaggedTensor [[1, 1], [89, 3, 1, 1, 1]]>

In [ ]:
reload(tf_model)
from tf_model import *

vocab_size = 1000
text_vec_layer = TextVectorization(max_tokens=vocab_size, ragged=True)
text_vec_layer.adapt(train_set.take(10).map(lambda reviews: reviews[0]))


embed_size = 128
tf.random.set_seed(42)

model = Sequential([
    Input([]),
    text_vec_layer,
    Embedding(vocab_size, embed_size),
    GRU(128),
    Dense(1, activation="sigmoid")
])

model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
history = model.fit(train_set.take(30), validation_data=valid_set.take(10), epochs=2)

Epoch 1/2


0it [00:00, ?it/s]

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, self, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, self, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, self, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, activation, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


...

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, metric, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


2026-02-25 02:12:11.012892: I external/local_xla/xla/service/service.cc:163] XLA service 0x71f5d8034d80 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2026-02-25 02:12:11.012925: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): Host, Default Version
2026-02-25 02:12:11.078064: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1771978331.491727    3334 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
30it [02:36,  5.22s/it]


    - loss: 0.695 - accuracy: 0.518
    - val_loss: 0.700 - val_accuracy: 0.450
    - Learning Rate: 0.001
Epoch 2/2


30it [02:41,  5.39s/it]


    - loss: 0.694 - accuracy: 0.508
    - val_loss: 0.692 - val_accuracy: 0.550
    - Learning Rate: 0.001
